# histogram based gradient boosting classification tree

## 0 Setup

In [1]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import PredefinedSplit

## 1 Load data

- Renskrevet version af 'hgbt-v1'.
- Data klagøres i '00_data-prep'

- I denne notebook er der derfor kun:
- Afsøgning af hyperparametre
- Retraining på den bedste config
- Evaluering på test data

In [2]:
DATA = '../data'

X_train = pd.read_csv(f'{DATA}/X_train.csv')
X_val   = pd.read_csv(f'{DATA}/X_val.csv')
X_test  = pd.read_csv(f'{DATA}/X_test.csv')

y_train = pd.read_csv(f'{DATA}/y_train.csv').squeeze('columns')
y_val   = pd.read_csv(f'{DATA}/y_val.csv').squeeze('columns')
y_test  = pd.read_csv(f'{DATA}/y_test.csv').squeeze('columns')

print('Shapes:')
print(f'  X_train: {X_train.shape}')
print(f'  X_val  : {X_val.shape}')
print(f'  X_test : {X_test.shape}')

print('\nKlassefordeling i y_train:')
print(y_train.value_counts(normalize=True).round(3).to_dict())
print(y_val.value_counts(normalize=True).round(3).to_dict())
print(y_test.value_counts(normalize=True).round(3).to_dict())

Shapes:
  X_train: (28941, 13)
  X_val  : (7236, 13)
  X_test : (9045, 13)

Klassefordeling i y_train:
{'<=50K': 0.752, '>50K': 0.248}
{'<=50K': 0.752, '>50K': 0.248}
{'<=50K': 0.752, '>50K': 0.248}


## 2 Data Prep

- Target laves binær (`>50K` → 1, `<=50K` → 0)
- `col_cat` definerer hvilke kolonner HGBT skal behandle som kategoriske i step 3

In [3]:
y_train_binary = (y_train == '>50K').astype(int)
y_val_binary   = (y_val   == '>50K').astype(int)
y_test_binary  = (y_test  == '>50K').astype(int)

col_cat = ['workclass', 'occupation', 'marital-status', 'relationship',
           'race', 'sex', 'native-country']

## 3 Model

- HGBT håndterer kategorier disse som kategorier`categorical_features`
- `early_stopping=True` + `max_iter=1000`: modellen stopper ved 1000 iterationer eller når intern validering ikke forbedres

In [4]:
hgb_clf = HistGradientBoostingClassifier(
    categorical_features=col_cat,
    max_iter=1000,
    early_stopping=True,
    random_state=42,
)

## 4 Training

- `.fit()` Denne metode som køre træningen
- `n_iter_` viser hvor mange runder der blev kørt før early stopping greb ind

In [5]:
hgb_clf.fit(X_train, y_train_binary)
print(f"runder kørt: {hgb_clf.n_iter_} / {hgb_clf.max_iter}")

runder kørt:{79}/{1000}


## 5 Evaluation


- Vi evaluerer baseline på `X_val`, ikke `X_test`. Test gemmes til step 7.
- Vi måler primært på F1

In [6]:
val_preds = hgb_clf.predict(X_val)
print("Val accuracy:", accuracy_score(y_val_binary, val_preds))
print(classification_report(y_val_binary, val_preds, target_names=['<=50K', '>50K']))

Val accuracy: 0.8729961304588171
              precision    recall  f1-score   support

       <=50K       0.89      0.94      0.92      5443
        >50K       0.79      0.66      0.72      1793

    accuracy                           0.87      7236
   macro avg       0.84      0.80      0.82      7236
weighted avg       0.87      0.87      0.87      7236



## 6 Hyperparameter Tuning

- Vi bruger `PredefinedSplit` så `X_val` aktivt bestemmer hvilke hyperparametre der vinder
- `scoring='f1'` fordi vi har ubalanceret data
- Efter tuning refitter GridSearchCV på `X_train + X_val` samlet, så den endelige model bruger al ikke-test data

In [7]:
# -1 = "denne række er træning", 0 = "denne række er validering"
split_index = [-1] * len(X_train) + [0] * len(X_val)
X_combined = pd.concat([X_train, X_val])
y_combined = pd.concat([y_train_binary, y_val_binary])

params = {
    'learning_rate':     [0.05, 0.1, 0.2],
    'max_leaf_nodes':    [10, 15, 31],
    'min_samples_leaf':  [50, 100, 250],
}

grid_search = GridSearchCV(
    hgb_clf, params, scoring='f1',
    cv=PredefinedSplit(split_index),
    n_jobs=-1, verbose=1,
)
grid_search.fit(X_combined, y_combined);

print(f"Best val F1: {grid_search.best_score_:.4f}")

Fitting 1 folds for each of 27 candidates, totalling 27 fits
Best val F1: 0.7215


In [8]:
grid_search.best_params_

{'learning_rate': 0.1, 'max_leaf_nodes': 15, 'min_samples_leaf': 100}

## 7 Prediction

- Vi evaluerer den tunede model på alle tre splits: train viser fit, val bekræfter tuning, test er det endelige tal.
- forskel mellem train og val F1 indikerer overfit.

In [9]:
train_preds = grid_search.predict(X_train)
test_preds  = grid_search.predict(X_test)

print(f"Train F1: {f1_score(y_train_binary, train_preds):.4f}")
print(f"Test F1 (final): {f1_score(y_test_binary, test_preds):.4f}")

Train F1: 0.7368
Test F1 (final): 0.7059
